[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/gate-close.ipynb)

# Gate close · the byte-confirm and the tight differentials

**Hardware:** any Colab TPU runtime. Run the first cell; if it installs anything, Runtime → Restart session, then Run all.

Two open bars, closed properly. Gate 02: the 268 MB spill estimate checked against XLA's own cost analysis (the compiler's independent accounting of bytes accessed), with a bandwidth cross-check from measured runtime. Gate 03: differentials computed the right way, f32 pipelines against the 1e-3 forward and 1e-2 grads bars, and the bf16 pipeline reported as relative error against an f32-computed reference instead of against a reference that rounds as much as the kernel does. One JSON blob at the end.


In [ ]:
!pip install -q -U "jax[tpu]"
import importlib.metadata as md
print("jax", md.version("jax"), "· libtpu", md.version("libtpu"))


In [ ]:
import os
os.environ.pop("TPU_LIBRARY_PATH", None)
import time, json, functools
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
CHIP = jax.devices()[0].device_kind if ON_TPU else "none"
RESULTS = {"chip": CHIP, "notebook": "gate-close", "results": {}}

def bench(fn, *args, reps=20):
    fn(*args).block_until_ready()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(*args).block_until_ready()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)) * 1e6

def record(key, value):
    RESULTS["results"][key] = value
    print(f"  -> {key} = {value}")

def maxerr(a, b):
    return float(jnp.abs(a.astype(jnp.float32) - b.astype(jnp.float32)).max())

if ON_TPU:
    pl.pallas_call(
        lambda x_ref, o_ref: o_ref.__setitem__(..., x_ref[...] + 1),
        out_shape=jax.ShapeDtypeStruct((8, 128), jnp.float32),
    )(jnp.zeros((8, 128), jnp.float32)).block_until_ready()
    print("pallas smoke test: OK")


## Gate 02 · the spill, confirmed in bytes


In [ ]:
# gate 02: the byte-confirm. Our estimate says the seq-8192 bf16 spill is
# 2 x S^2 x 2 bytes = 268.4 MB of round-trip traffic. XLA's cost analysis
# is the compiler's own accounting of bytes accessed; if the two agree
# within the 20% bar, the estimate is confirmed by an independent model.
def naive_attention(q, k, v):
    s = q @ k.T
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    return (p / jnp.sum(p, axis=-1, keepdims=True)) @ v

if ON_TPU:
    S, D = 8192, 128
    args = [jax.ShapeDtypeStruct((S, D), jnp.bfloat16)] * 3
    compiled = jax.jit(naive_attention).lower(*args).compile()
    ca = compiled.cost_analysis()
    if isinstance(ca, (list, tuple)):
        ca = ca[0]
    xla_bytes = float(ca.get("bytes accessed", 0.0))
    est_spill = 2 * S * S * 2                      # write + read, bf16
    io_bytes = 4 * S * D * 2                       # q, k, v in, o out
    est_total = est_spill + io_bytes

    # bandwidth cross-check: measured runtime x chip bandwidth bounds the
    # traffic a memory-bound op could have moved
    xs = [jax.random.normal(jax.random.key(i), (S, D), jnp.bfloat16) for i in range(3)]
    us = bench(jax.jit(naive_attention), *xs)
    bw = 1.6e12 if "v6" in CHIP.lower() else 8.2e11
    bw_implied = us * 1e-6 * bw

    record("gate02/byte_confirm", {
        "estimate_spill_mb": round(est_spill / 1e6, 1),
        "estimate_total_mb": round(est_total / 1e6, 1),
        "xla_bytes_accessed_mb": round(xla_bytes / 1e6, 1),
        "ratio_vs_estimate": round(xla_bytes / est_total, 3) if xla_bytes else None,
        "within_20pct": bool(xla_bytes and 0.8 <= xla_bytes / est_total <= 1.2),
        "runtime_us": round(us, 1),
        "bandwidth_implied_mb_ceiling": round(bw_implied / 1e6, 1),
        "method": "xla cost_analysis bytes accessed + bandwidth cross-check",
    })


## Gate 03 · forward differentials, done right


In [ ]:
# gate 03: the differentials, done right. The bars are 1e-3 forward and
# 1e-2 grads. The earlier 0.25 came from comparing bf16 against a reference
# that itself computes in bf16; here the f32 pipelines meet the bars, and
# the bf16 pipeline is reported as relative error against an f32 reference.
def flash_kernel(q_ref, k_ref, v_ref, o_ref, *, block_kv):
    q = q_ref[...].astype(jnp.float32)
    n_kv = k_ref.shape[0]
    def step(j, state):
        m, l, acc = state
        kb = k_ref[pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        vb = v_ref[pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        s = q @ kb.T
        m_new = jnp.maximum(m, jnp.max(s, axis=-1))
        alpha = jnp.exp(m - m_new)
        p = jnp.exp(s - m_new[:, None])
        return m_new, l * alpha + jnp.sum(p, axis=-1), acc * alpha[:, None] + p @ vb
    m0 = jnp.full((q.shape[0],), -jnp.inf, jnp.float32)
    l0 = jnp.zeros((q.shape[0],), jnp.float32)
    acc0 = jnp.zeros((q.shape[0], v_ref.shape[1]), jnp.float32)
    m, l, acc = jax.lax.fori_loop(0, n_kv // block_kv, step, (m0, l0, acc0))
    o_ref[...] = (acc / l[:, None]).astype(o_ref.dtype)

def flash(q, k, v, block_q=512, block_kv=1024):
    sq, d = q.shape
    return pl.pallas_call(
        functools.partial(flash_kernel, block_kv=block_kv),
        grid=(sq // block_q,),
        in_specs=[pl.BlockSpec((block_q, d), lambda i: (i, 0)),
                  pl.BlockSpec(k.shape, lambda i: (0, 0)),
                  pl.BlockSpec(v.shape, lambda i: (0, 0))],
        out_specs=pl.BlockSpec((block_q, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
    )(q, k, v)

def ref_attention_f32(q, k, v):
    s = (q.astype(jnp.float32) @ k.astype(jnp.float32).T)
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    return (p / jnp.sum(p, axis=-1, keepdims=True)) @ v.astype(jnp.float32)

if ON_TPU:
    S, D = 8192, 128
    qf = jax.random.normal(jax.random.key(0), (S, D), jnp.float32)
    kf = jax.random.normal(jax.random.key(1), (S, D), jnp.float32)
    vf = jax.random.normal(jax.random.key(2), (S, D), jnp.float32)

    fwd_f32 = maxerr(jax.jit(flash)(qf, kf, vf), jax.jit(ref_attention_f32)(qf, kf, vf))

    qb, kb_, vb_ = (x.astype(jnp.bfloat16) for x in (qf, kf, vf))
    ref_from_bf16 = jax.jit(ref_attention_f32)(qb, kb_, vb_)
    got_bf16 = jax.jit(flash)(qb, kb_, vb_)
    abs_bf16 = maxerr(got_bf16, ref_from_bf16)
    rel_bf16 = abs_bf16 / float(jnp.abs(ref_from_bf16).max())

    record("gate03/forward", {
        "f32_max_err": round(fwd_f32, 8),
        "f32_bar_1e3": bool(fwd_f32 < 1e-3),
        "bf16_abs_err_vs_f32_ref": round(abs_bf16, 5),
        "bf16_rel_err": round(rel_bf16, 5),
    })


## Gate 03 · gradient differentials


In [ ]:
# gate 03b: gradients. custom_vjp against autodiff, both in f32, against
# the 1e-2 bar; then the bf16 pipeline's grads as relative error against
# the f32-computed gradients of the same bf16 inputs.
def attention_ref(q, k, v):
    s = (q @ k.T).astype(jnp.float32)
    p = jax.nn.softmax(s, axis=-1)
    return (p.astype(q.dtype)) @ v

@jax.custom_vjp
def attention_cv(q, k, v):
    return attention_ref(q, k, v)

def fwd(q, k, v):
    s = (q @ k.T).astype(jnp.float32)
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    l = jnp.sum(p, axis=-1, keepdims=True)
    o = ((p / l).astype(q.dtype)) @ v
    return o, (q, k, v, o, m, l)

def bwd(res, do):
    q, k, v, o, m, l = res
    p = (jnp.exp((q @ k.T).astype(jnp.float32) - m) / l).astype(q.dtype)
    dv = p.T @ do
    d = jnp.sum((do * o).astype(jnp.float32), axis=-1, keepdims=True).astype(q.dtype)
    dp = do @ v.T
    ds = p * (dp - d)
    return ds @ k, ds.T @ q, dv

attention_cv.defvjp(fwd, bwd)

if ON_TPU:
    def loss(fn):
        return lambda *a: jnp.sum(jnp.tanh(fn(*a).astype(jnp.float32)))

    qf = jax.random.normal(jax.random.key(0), (512, 128), jnp.float32)
    kf = jax.random.normal(jax.random.key(1), (1024, 128), jnp.float32)
    vf = jax.random.normal(jax.random.key(2), (1024, 128), jnp.float32)
    g1 = jax.grad(loss(attention_cv), argnums=(0, 1, 2))(qf, kf, vf)
    g2 = jax.grad(loss(attention_ref), argnums=(0, 1, 2))(qf, kf, vf)
    errs_f32 = {n: maxerr(x, y) for n, x, y in zip("qkv", g1, g2)}

    qb, kb_, vb_ = (x.astype(jnp.bfloat16) for x in (qf, kf, vf))
    gb = jax.grad(loss(attention_cv), argnums=(0, 1, 2))(qb, kb_, vb_)
    gf = jax.grad(loss(attention_ref), argnums=(0, 1, 2))(qf, kf, vf)
    rel_bf16 = {n: maxerr(x, y) / float(jnp.abs(y).max()) for n, x, y in zip("qkv", gb, gf)}

    record("gate03/grads", {
        "f32_max_err": {n: round(v, 8) for n, v in errs_f32.items()},
        "f32_bar_1e2": bool(max(errs_f32.values()) < 1e-2),
        "bf16_rel_err_vs_f32": {n: round(v, 4) for n, v in rel_bf16.items()},
    })


## The blob


In [ ]:
print("=" * 60)
print("GATE CLOSE RESULTS · paste this whole blob back")
print("=" * 60)
print(json.dumps(RESULTS, indent=1))
